# # 使用贝叶斯优化得到的最佳超参数进行模型训练
#
# 本notebook旨在利用之前贝叶斯优化得到的最佳超参数组合，重新进行一次完整的模型训练、验证和测试。
# 我们将尽可能复用项目中的现有代码模块。


In [ ]:
# 单元格1: 导入模块
import os
import sys
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score
import random

# 添加PyTorch序列化安全设置，避免加载模型时报错
try:
    safe_globals = [
        np.dtype,
        np.core.multiarray.scalar,
        np.ndarray,
        np.generic,
        np.float64,
        np.float32,
        np.int64,
        np.int32
    ]
    torch.serialization.add_safe_globals(safe_globals)
    print("✅ 已添加numpy类型到PyTorch安全全局变量列表")
except Exception as e:
    print(f"⚠️ 添加安全全局变量时出错 (可忽略): {e}")

# 导入项目中的模块
from config import load_config, save_config, is_mat_format
from data import load_data
from data.mat_loader import validate_filtered_data
from models import get_model
from utils.metrics import calculate_class_weights, evaluate_model
from utils.visualization import visualize_training_curves, visualize_confusion_matrix
from utils.model_io import save_model_with_architecture


In [ ]:
# 单元格2: 配置设置
cfg = load_config()

# 设置 MAT 文件路径（请确保该路径有效）
mat_file_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat"
if not os.path.exists(mat_file_path):
    print(f"错误: MAT文件不存在: {mat_file_path}")
    print("请修改为正确的路径")
    raise FileNotFoundError(f"MAT文件未找到: {mat_file_path}")

cfg['mat_file_path'] = mat_file_path

# 设置患者ID分割（与 main.py 保持一致）
cfg['dataset_split'] = {
    'train_patients': [28, 5, 25, 30, 34, 32, 33, 11, 12, 20, 29, 17, 37, 7, 26, 1, 36, 14, 19, 3, 35, 31, 22, 8],
    'val_patients': [4, 24, 9, 15, 16, 18, 2],
    'test_patients': [38, 6, 21, 13, 10, 23, 27]
}

# 清空原版数据目录路径，确保使用MAT格式
cfg['data_dirs'] = {
    'train_dir': None,
    'test_dir': None,
    'val_dir': None
}

# 设置实验名称
cfg['experiment_name'] = f"BestHyperparams_MAT_{time.strftime('%Y%m%d_%H%M%S')}"

# 确保关键配置与main.py一致
cfg['test_size'] = 0.01  # MAT格式的测试集分割比例
cfg['use_old_zipfile_serialization'] = True
cfg['norm'] = True  # 确保标准化开启
cfg['apply_pca'] = False  # 确保PCA设置正确
cfg['n_pca'] = 0

print(f"确认配置 mat_file_path: {cfg.get('mat_file_path')}")
print(f"数据格式检测: {'MAT格式' if is_mat_format(cfg) else '原版格式'}")
print(f"实验名称: {cfg['experiment_name']}")

In [ ]:
# 单元格3: 最佳超参数设置
best_hyperparams_from_bayesian = {
    'learning_rate': 9.191261837889327e-05,
    'weight_decay': 0.0005175833650131985,
    'optimizer': 'adamw',
    'dropout_rate': 0.19317216698770392,
    'activation': 'gelu',
    'lr_scheduler': 'step',
    'model_type': 'base_mlp',
    'step_size': 4,
    'step_gamma': 0.1590396939768399,
    'layer_sizes_idx': 0,  # 对应 [4096, 4096, 4096, 4096]
}

print("🔧 应用贝叶斯优化的最佳超参数...")

# 更新配置 - 按照main.py的参数映射方式
for key, value in best_hyperparams_from_bayesian.items():
    if key == 'learning_rate':
        cfg['lr'] = value
    elif key == 'lr_scheduler':
        cfg['lr_scheduler_type'] = value
        cfg['use_lr_scheduler'] = True
    elif key == 'step_size':
        # 注意：需要保存为lr_milestones的格式，但StepLR使用step_size
        cfg['lr_step_size'] = value  # 为StepLR准备
        cfg['lr_milestones'] = [value, value*2]  # 为MultiStepLR准备
    elif key == 'step_gamma':
        cfg['lr_gamma'] = value
    elif key == 'layer_sizes_idx':
        # 根据索引设置隐藏层配置 - 与optimization.py中的定义一致
        layer_sizes_options = [
            [4096, 4096, 4096, 4096],  # idx=0
            [3072, 3072, 3072, 3072],  # idx=1  
            [2048, 2048, 2048, 2048],  # idx=2
        ]
        if value < len(layer_sizes_options):
            cfg['hidden_units'] = layer_sizes_options[value]
        else:
            cfg['hidden_units'] = layer_sizes_options[0]  # 默认使用第一个
    else:
        cfg[key] = value

# 确保其他必要参数与main.py一致
cfg['epochs'] = cfg.get('epochs', 30)
cfg['batch_size'] = cfg.get('batch_size', 128)
cfg['val_epochs'] = cfg.get('val_epochs', 3)
cfg['save_checkpoints'] = True
cfg['model_name'] = cfg.get('model_name', 'BrainVoxel_102Class_MLP')
cfg['dataset_name'] = cfg.get('dataset_name', 'BrainVoxel')

print("✅ 最佳超参数配置完成:")
print(f"  模型类型: {cfg['model_type']}")
print(f"  隐藏层: {cfg['hidden_units']}")
print(f"  学习率: {cfg['lr']:.2e}")
print(f"  权重衰减: {cfg['weight_decay']:.2e}")
print(f"  优化器: {cfg['optimizer']}")
print(f"  激活函数: {cfg['activation']}")
print(f"  Dropout率: {cfg['dropout_rate']:.4f}")
print(f"  学习率调度器: {cfg['lr_scheduler_type']}")
if cfg['lr_scheduler_type'] == 'step':
    print(f"    Step Size: {cfg.get('lr_step_size', 4)}")
    print(f"    Gamma: {cfg.get('lr_gamma', 0.1)}")

In [ ]:
# 单元格4: 环境设置（与main.py保持一致）

def setup_environment(config):
    """设置环境，包括随机种子和设备（与main.py一致）"""
    # 设置随机种子
    random.seed(config['random_seed'])
    torch.manual_seed(config['random_seed'])
    torch.cuda.manual_seed(config['random_seed'])
    torch.cuda.manual_seed_all(config['random_seed'])
    np.random.seed(config['random_seed'])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # 设置设备
    device = torch.device(f"cuda:{config['device']}" if config['device'] >= 0 and torch.cuda.is_available() else "cpu")
    
    print(f"🎯 随机种子设置为: {config['random_seed']}")
    print(f"🖥️ 使用设备: {device}")
    
    return device

device = setup_environment(cfg)

# 创建保存目录
save_dir = os.path.join(cfg.get('save_dir', './results'), cfg['experiment_name'])
cfg['save_dir'] = save_dir
os.makedirs(save_dir, exist_ok=True)
print(f"📁 结果保存目录: {save_dir}")


In [ ]:
# 单元格5: 加载数据（完全按照main.py的流程）
print(f"🔄 准备使用MAT格式加载数据: {cfg['mat_file_path']}...")

# 确保MAT文件存在
if not os.path.exists(cfg['mat_file_path']):
    raise FileNotFoundError(f"MAT文件不存在: {cfg['mat_file_path']}")

# 加载数据 - 使用统一接口，与main.py完全一致
dataset_dict, train_loader, val_loader, test_loader = load_data(cfg, mode='train')

# 🔍 数据完整性验证 - 与main.py完全一致
print("\n🔍 验证数据完整性（与main.py一致）:")
validate_filtered_data(dataset_dict['train_samples'], dataset_dict['train_labels'], "最终训练数据")
validate_filtered_data(dataset_dict['val_samples'], dataset_dict['val_labels'], "最终验证数据")
validate_filtered_data(dataset_dict['test_samples'], dataset_dict['test_labels'], "最终测试数据")

# 更新配置中的特征维度
cfg['feature_dim'] = dataset_dict['feature_dim']
# cfg['num_class'] 已由数据加载自动设置

# 🔍 验证标准化状态
print("\n📊 验证StandardScaler标准化:")
scaler = dataset_dict.get('scaler', None)
if scaler is not None:
    print("✅ 检测到StandardScaler对象:")
    if hasattr(scaler, 'mean_') and hasattr(scaler, 'scale_'):
        print(f"  ✓ Scaler均值范围: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]")
        print(f"  ✓ Scaler缩放范围: [{scaler.scale_.min():.4f}, {scaler.scale_.max():.4f}]")
    
    # 验证实际标准化效果
    train_mean = np.mean(dataset_dict['train_samples'], axis=0)
    train_std = np.std(dataset_dict['train_samples'], axis=0)
    print(f"  ✓ 实际数据均值范围: [{train_mean.min():.4f}, {train_mean.max():.4f}]")
    print(f"  ✓ 实际数据标准差范围: [{train_std.min():.4f}, {train_std.max():.4f}]")
    
    if abs(train_mean.mean()) < 0.1 and abs(train_std.mean() - 1.0) < 0.2:
        print("  ✅ 三个数据集已正确通过StandardScaler标准化（均值≈0，标准差≈1）")
    else:
        print("  ⚠️ 标准化可能不完整")
else:
    print("⚠️ 未找到StandardScaler对象")

# 🔍 关键修正：验证标签格式和背景过滤状态
print("\n🎯 验证标签格式和背景过滤状态:")

# 🔧 关键修正：正确处理标签格式检测
def get_label_indices(labels):
    """统一的标签格式处理函数"""
    if len(labels.shape) > 1 and labels.shape[1] > 1:
        # one-hot编码格式，转换为索引
        print("  📋 检测到one-hot编码格式标签")
        return np.argmax(labels, axis=1)
    else:
        # 已经是索引格式
        print("  📋 检测到索引格式标签")
        return labels.flatten()

# 检查训练集标签
train_label_indices = get_label_indices(dataset_dict['train_labels'])
val_label_indices = get_label_indices(dataset_dict['val_labels'])
test_label_indices = get_label_indices(dataset_dict['test_labels'])

print(f"  ✓ 训练集标签形状: {dataset_dict['train_labels'].shape} -> 索引形状: {train_label_indices.shape}")
print(f"  ✓ 验证集标签形状: {dataset_dict['val_labels'].shape} -> 索引形状: {val_label_indices.shape}")
print(f"  ✓ 测试集标签形状: {dataset_dict['test_labels'].shape} -> 索引形状: {test_label_indices.shape}")

# 检查标签范围
unique_train_labels = np.unique(train_label_indices)
unique_val_labels = np.unique(val_label_indices)
unique_test_labels = np.unique(test_label_indices)

print(f"  ✓ 训练集标签范围: {unique_train_labels.min()} - {unique_train_labels.max()}")
print(f"  ✓ 验证集标签范围: {unique_val_labels.min()} - {unique_val_labels.max()}")
print(f"  ✓ 测试集标签范围: {unique_test_labels.min()} - {unique_test_labels.max()}")

# 🔧 关键检查：验证背景过滤和标签映射
all_unique_labels = np.unique(np.concatenate([unique_train_labels, unique_val_labels, unique_test_labels]))
print(f"  ✓ 所有数据集标签范围: {all_unique_labels.min()} - {all_unique_labels.max()}")

# 检查是否包含背景标签
if 0 in all_unique_labels:
    print("  ⚠️ 警告: 检测到标签0，可能仍包含背景")
    # 进一步检查标签分布
    zero_count_train = np.sum(train_label_indices == 0)
    zero_count_val = np.sum(val_label_indices == 0)
    zero_count_test = np.sum(test_label_indices == 0)
    print(f"    训练集标签0数量: {zero_count_train}")
    print(f"    验证集标签0数量: {zero_count_val}")
    print(f"    测试集标签0数量: {zero_count_test}")
else:
    print("  ✅ 确认: 背景标签0已在加载阶段过滤")

# 检查标签连续性和期望范围
if all_unique_labels.min() >= 1 and all_unique_labels.max() <= 102:
    print("  ✅ 标签范围正确: 1-102 (原始标签，背景已过滤)")
    print("  📝 注意: BrainVoxelMatDataset会将1-102映射为0-101用于训练")
elif all_unique_labels.min() >= 0 and all_unique_labels.max() <= 101:
    print("  ✅ 标签范围正确: 0-101 (已映射用于训练)")
else:
    print(f"  ⚠️ 警告: 标签范围异常 [{all_unique_labels.min()}, {all_unique_labels.max()}]")

# 检查标签连续性
expected_range_original = set(range(1, 103))  # 1-102
expected_range_mapped = set(range(0, 102))    # 0-101
actual_labels = set(all_unique_labels)

if actual_labels.issubset(expected_range_original):
    missing_labels = expected_range_original - actual_labels
    print("  📊 使用原始标签格式 (1-102)")
elif actual_labels.issubset(expected_range_mapped):
    missing_labels = expected_range_mapped - actual_labels
    print("  📊 使用映射标签格式 (0-101)")
else:
    missing_labels = set()
    print("  ❓ 标签格式需要进一步确认")

if missing_labels and len(missing_labels) < 50:
    print(f"  ℹ️ 缺失的标签: {sorted(list(missing_labels))[:10]}{'...' if len(missing_labels) > 10 else ''}")
elif len(missing_labels) >= 50:
    print(f"  ℹ️ 缺失标签数量: {len(missing_labels)}")

print(f"\n✅ 数据加载完成:")
print(f"  📁 数据格式: {'MAT格式' if is_mat_format(cfg) else '原版格式'}")
print(f"  📊 训练集样本数: {len(dataset_dict['train_samples'])}")
print(f"  📊 验证集样本数: {len(dataset_dict['val_samples'])}")
print(f"  📊 测试集样本数: {len(dataset_dict['test_samples'])}")
print(f"  🔢 特征维度: {cfg['feature_dim']}")
print(f"  🏷️ 类别数量: {cfg['num_class']}")
print(f"  ⚙️ 数据处理状态:")
print(f"    • 背景像素已在加载阶段过滤 ✓")
print(f"    • StandardScaler标准化已应用 ✓")
print(f"    • 患者ID分割已应用 ✓")
print(f"    • 标签格式已处理 ✓")

# 🔍 额外验证：检查数据加载路径
print(f"\n🛠️ 数据加载验证:")
print(f"  MAT文件路径: {cfg.get('mat_file_path', 'None')}")
print(f"  是否使用MAT格式: {is_mat_format(cfg)}")
if 'data_dirs' in cfg and any(cfg['data_dirs'].values()):
    print("  ⚠️ 警告: 检测到原版数据目录配置，但应优先使用MAT格式")

# 🔧 新增：验证DataLoader中的标签格式
print(f"\n🔍 验证DataLoader中的标签格式:")
try:
    # 获取一个批次来验证标签格式
    sample_batch = next(iter(train_loader))
    sample_data, sample_labels = sample_batch
    print(f"  DataLoader输出标签形状: {sample_labels.shape}")
    print(f"  DataLoader输出标签范围: {sample_labels.min().item()} - {sample_labels.max().item()}")
    print(f"  DataLoader输出标签类型: {sample_labels.dtype}")
    
    # 验证BrainVoxelMatDataset的标签映射是否正确
    if sample_labels.min().item() >= 0 and sample_labels.max().item() <= 101:
        print("  ✅ DataLoader标签格式正确: 0-101 (适用于CrossEntropyLoss)")
    else:
        print(f"  ⚠️ DataLoader标签范围异常: [{sample_labels.min().item()}, {sample_labels.max().item()}]")
        
except Exception as e:
    print(f"  ⚠️ 验证DataLoader标签格式时出错: {e}")

In [ ]:
# 单元格6: 初始化模型
# 初始化模型
print(f"🔧 创建模型: {cfg['model_type']}...")

model = get_model(
    model_type=cfg['model_type'],
    input_dim=cfg['feature_dim'],
    hidden_dims=cfg['hidden_units'],
    num_classes=cfg['num_class'],
    dropout_rate=cfg['dropout_rate'],
    activation=cfg['activation']
)
model.to(device)

print(f"✅ 模型创建完成:")
print(f"  模型类型: {cfg['model_type']}")
print(f"  输入维度: {cfg['feature_dim']}")
print(f"  输出类别数: {cfg['num_class']}")
print(f"  隐藏层配置: {cfg['hidden_units']}")
print(f"  激活函数: {cfg['activation']}")
print(f"  Dropout率: {cfg['dropout_rate']}")

# 计算参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  总参数量: {total_params:,}")
print(f"  可训练参数量: {trainable_params:,}")


In [ ]:
# 单元格7: 定义优化器、学习率调度器和损失函数
print("⚖️ 计算类别权重...")

# 🔧 关键修正：确保权重计算使用正确的标签格式
print("🔍 验证权重计算的标签格式...")

# 检查训练标签格式并进行权重计算
train_labels_for_weights = dataset_dict['train_labels']
print(f"权重计算使用的标签形状: {train_labels_for_weights.shape}")

if len(train_labels_for_weights.shape) > 1 and train_labels_for_weights.shape[1] > 1:
    print("权重计算: 检测到one-hot编码，calculate_class_weights会自动处理")
else:
    print("权重计算: 检测到索引格式")

# 🔧 关键修正：使用与main.py完全一致的类别权重计算
class_weights = calculate_class_weights(
    train_labels_for_weights,  # 可能是one-hot或索引格式
    cfg['num_class']
).to(device)

print(f"类别权重计算完成:")
print(f"  权重数量: {len(class_weights)}")
print(f"  权重范围: [{class_weights.min().item():.4f}, {class_weights.max().item():.4f}]")
print(f"  零权重类别数: {(class_weights == 0).sum().item()}")

# 🔧 关键修正：与main.py完全一致 - 使用类别权重，但不使用ignore_index（因为背景已过滤）
print("📊 创建损失函数: CrossEntropyLoss (使用类别权重，背景已过滤)")
criterion = nn.CrossEntropyLoss(weight=class_weights)  # 背景已过滤，不需要ignore_index

print(f"🚀 创建优化器: {cfg['optimizer']}")
if cfg['optimizer'].lower() == 'adam':
    optimizer = optim.Adam(
        model.parameters(), 
        lr=cfg['lr'], 
        weight_decay=cfg['weight_decay']
    )
elif cfg['optimizer'].lower() == 'adamw':
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=cfg['lr'], 
        weight_decay=cfg['weight_decay']
    )
else:
    raise ValueError(f"不支持的优化器: {cfg['optimizer']}")

# 创建学习率调度器 - 与main.py完全一致
lr_scheduler = None
if cfg['use_lr_scheduler']:
    print(f"📈 创建学习率调度器: {cfg['lr_scheduler_type']}")
    
    if cfg['lr_scheduler_type'].lower() == 'cosine':
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg['epochs']
        )
    elif cfg['lr_scheduler_type'].lower() == 'multistep':
        lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer, 
            milestones=cfg.get('lr_milestones', [10, 20]), 
            gamma=cfg.get('lr_gamma', 0.1)
        )
    elif cfg['lr_scheduler_type'].lower() == 'step':
        lr_scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, 
            step_size=cfg.get('lr_step_size', 4), 
            gamma=cfg.get('lr_gamma', 0.1)
        )
    elif cfg['lr_scheduler_type'].lower() == 'plateau':
        lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=cfg.get('lr_gamma', 0.1), 
            patience=5, verbose=True
        )
    else:
        print(f"⚠️ 未知的学习率调度器类型: {cfg['lr_scheduler_type']}. 不使用调度器。")
        lr_scheduler = None
else:
    print("🚫 不使用学习率调度器")

# 🔧 关键修正：标准化参数处理 - 与main.py train_and_evaluate函数完全一致
print("📊 处理标准化参数...")

# MAT格式数据已经通过StandardScaler标准化，从dataset_dict中获取scaler
scaler = dataset_dict.get('scaler', None)
normalization_params = None

if scaler is not None:
    print("✅ 检测到StandardScaler对象（MAT数据已标准化）")
    
    # 🔧 关键修正：从StandardScaler提取参数供predict.py使用 - 与main.py一致
    if hasattr(scaler, 'mean_') and hasattr(scaler, 'scale_'):
        # StandardScaler的scale_ = 1/std，所以std = 1/scale_
        std_values = 1.0 / scaler.scale_
        normalization_params = {
            'mean': scaler.mean_.tolist(),
            'std': std_values.tolist()
        }
        print(f"  均值范围: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]")
        print(f"  标准差范围: [{std_values.min():.4f}, {std_values.max():.4f}]")
        print("  ✅ 标准化参数已提取（用于预测阶段）")
    else:
        print("  ⚠️ StandardScaler对象不完整")
        
    # 将scaler保存到配置中
    cfg['scaler'] = scaler
else:
    print("⚠️ 未找到StandardScaler对象")
    
    # 如果没有scaler但需要标准化参数，尝试从训练数据计算
    if cfg.get('norm', True):
        print("⚠️ 尝试从训练数据计算标准化参数...")
        train_samples = dataset_dict['train_samples']
        mean = np.mean(train_samples, axis=0)
        std = np.std(train_samples, axis=0)
        std[std == 0] = 1e-10  # 避免除零
        
        normalization_params = {
            'mean': mean.tolist(),
            'std': std.tolist()
        }
        print(f"  从训练数据计算的均值范围: [{mean.min():.4f}, {mean.max():.4f}]")
        print(f"  从训练数据计算的标准差范围: [{std.min():.4f}, {std.max():.4f}]")

# 验证数据已经被标准化
print("\n🔍 验证数据标准化状态:")
train_mean = np.mean(dataset_dict['train_samples'], axis=0)
train_std = np.std(dataset_dict['train_samples'], axis=0)
print(f"  训练数据均值范围: [{train_mean.min():.4f}, {train_mean.max():.4f}]")
print(f"  训练数据标准差范围: [{train_std.min():.4f}, {train_std.max():.4f}]")

if abs(train_mean.mean()) < 0.1 and abs(train_std.mean() - 1.0) < 0.1:
    print("  ✅ 数据已正确标准化（均值≈0，标准差≈1）")
else:
    print("  ⚠️ 数据可能未正确标准化")

# 打印训练配置摘要
print(f"\n📋 训练配置摘要:")
print(f"  优化器: {cfg['optimizer']}")
print(f"  学习率: {cfg['lr']:.2e}")
print(f"  权重衰减: {cfg['weight_decay']:.2e}")
print(f"  学习率调度器: {cfg['lr_scheduler_type'] if lr_scheduler else 'None'}")
if lr_scheduler and cfg['lr_scheduler_type'].lower() == 'step':
    print(f"    Step Size: {cfg.get('lr_step_size', 4)}")
    print(f"    Gamma: {cfg.get('lr_gamma', 0.1)}")
print(f"  损失函数: CrossEntropyLoss (使用类别权重)")
print(f"  批处理大小: {cfg['batch_size']}")
print(f"  训练轮数: {cfg['epochs']}")
print(f"  验证频率: 每 {cfg['val_epochs']} 轮")
print(f"  🔧 关键：背景像素已过滤，DataLoader输出标签0-101，模型输出102个类别")

In [ ]:
# 单元格8: 训练与验证循环
print(f"🚀 开始训练模型: {cfg.get('model_name', 'BrainVoxelMLP')}")
print(f"  总轮数: {cfg['epochs']}")
print(f"  验证频率: 每 {cfg['val_epochs']} 轮")
print(f"  数据格式: MAT格式")
print(f"  类别数量: {cfg['num_class']} (背景已过滤)")

# 初始化训练结果记录
training_results = {
    'loss_list': [],
    'acc_list': [],
    'f1_macro_list': [],
    'val_epoch_list': [],
    'val_acc_list': [],
    'val_f1_macro_list': [],
    'val_kappa_list': [],
    'val_balanced_acc_list': [],
    'lr_list': []
}

best_val_f1 = 0.0
best_model_path = os.path.join(save_dir, f"{cfg['experiment_name']}_best_model.pth")

# 开始训练循环
train_start_time = time.time()

for epoch in range(1, cfg['epochs'] + 1):
    current_lr = optimizer.param_groups[0]['lr']
    training_results['lr_list'].append(current_lr)

    model.train()
    epoch_loss = 0
    train_preds_epoch = []
    train_targets_epoch = []

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Train]", leave=False)
    for batch_idx, (data, target) in enumerate(progress_bar):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds = torch.argmax(output, dim=1)
        train_preds_epoch.extend(preds.cpu().numpy())
        train_targets_epoch.extend(target.cpu().numpy())

        if batch_idx % 100 == 0:
            progress_bar.set_postfix(loss=loss.item(), lr=current_lr)

    avg_epoch_loss = epoch_loss / len(train_loader)
    epoch_train_acc = accuracy_score(train_targets_epoch, train_preds_epoch)
    epoch_train_f1 = f1_score(train_targets_epoch, train_preds_epoch, average='macro', zero_division=0)

    training_results['loss_list'].append(avg_epoch_loss)
    training_results['acc_list'].append(epoch_train_acc)
    training_results['f1_macro_list'].append(epoch_train_f1)

    print(f"Epoch {epoch}/{cfg['epochs']} - Loss: {avg_epoch_loss:.4f}, Acc: {epoch_train_acc:.4f}, F1: {epoch_train_f1:.4f}, LR: {current_lr:.2e}")

    # 验证阶段
    if epoch % cfg['val_epochs'] == 0 or epoch == cfg['epochs']:
        model.eval()
        val_preds_epoch = []
        val_targets_epoch = []

        with torch.no_grad():
            progress_bar_val = tqdm(val_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Val]", leave=False)
            for data, target in progress_bar_val:
                data, target = data.to(device), target.to(device)
                output = model(data)
                preds = torch.argmax(output, dim=1)
                val_preds_epoch.extend(preds.cpu().numpy())
                val_targets_epoch.extend(target.cpu().numpy())

        val_acc = accuracy_score(val_targets_epoch, val_preds_epoch)
        val_f1_macro = f1_score(val_targets_epoch, val_preds_epoch, average='macro', zero_division=0)
        val_kappa = cohen_kappa_score(val_targets_epoch, val_preds_epoch)
        val_balanced_acc = balanced_accuracy_score(val_targets_epoch, val_preds_epoch)

        training_results['val_epoch_list'].append(epoch)
        training_results['val_acc_list'].append(val_acc)
        training_results['val_f1_macro_list'].append(val_f1_macro)
        training_results['val_kappa_list'].append(val_kappa)
        training_results['val_balanced_acc_list'].append(val_balanced_acc)

        train_val_f1_diff = epoch_train_f1 - val_f1_macro
        print(f"  Validation - Acc: {val_acc:.4f}, F1: {val_f1_macro:.4f}, Kappa: {val_kappa:.4f}, Bal_Acc: {val_balanced_acc:.4f}")
        print(f"  Train vs Val F1 Diff: {train_val_f1_diff:.4f} (正值可能表示过拟合)")

        # 🔧 关键修正：与main.py完全一致的模型保存逻辑
        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            training_info = {
                'epoch': epoch,
                'loss_list': training_results['loss_list'],
                'acc_list': training_results['acc_list'],
                'f1_macro_list': training_results['f1_macro_list'],
                'val_acc_list': training_results['val_acc_list'],
                'val_epoch_list': training_results['val_epoch_list'],
                'val_f1_macro_list': training_results['val_f1_macro_list'],
                'val_kappa_list': training_results['val_kappa_list'],
                'val_balanced_acc_list': training_results['val_balanced_acc_list'],
                'lr_list': training_results['lr_list'],
                'last_epoch': epoch,
                'train_time': time.time() - train_start_time,
                'num_classes': cfg.get('num_class'),
                'background_filtered': True  # ← 与main.py一致的标志
            }
            
            try:
                # 🔧 关键修正：与main.py完全一致的保存方式
                _, saved_path = save_model_with_architecture(
                    model=model,
                    optimizer=optimizer,
                    config=cfg,
                    training_info=training_info,
                    normalization_params=normalization_params,
                    save_path=best_model_path,
                    lr_scheduler=lr_scheduler,
                    use_old_zipfile_serialization=cfg.get('use_old_zipfile_serialization', True)
                    # 注意：scaler会从cfg中自动获取，不需要单独传递
                )
                print(f"   ✅ 最佳模型已保存: {os.path.basename(saved_path)} (F1: {best_val_f1:.4f})")
            except Exception as e:
                print(f"   ❌ 模型保存失败: {e}")
                # 备用保存方法
                torch.save({
                    'epoch': epoch,
                    'state_dict': model.state_dict(), 
                    'optimizer': optimizer.state_dict(),
                    'val_f1_macro': val_f1_macro,
                    'config': cfg,
                    'training_info': training_info,
                    'normalization_params': normalization_params,  # 🔧 确保保存标准化参数
                    'created_with': f'PyTorch {torch.__version__}',
                    'save_format_version': 1.0,
                    'numpy_version': f'{np.__version__}'
                }, best_model_path, _use_new_zipfile_serialization=not cfg.get('use_old_zipfile_serialization', True))
                print(f"   ✅ 使用备用方法保存模型: {os.path.basename(best_model_path)}")

    # 🔧 关键修正：学习率调度器更新逻辑 - 与main.py完全一致
    if lr_scheduler is not None:
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            # ReduceLROnPlateau需要在验证后才更新
            if epoch % cfg['val_epochs'] == 0 or epoch == cfg['epochs']:
                lr_scheduler.step(val_f1_macro)  # 使用验证F1指导学习率调度
        else:
            # 其他调度器每个epoch更新
            lr_scheduler.step()

total_train_time = time.time() - train_start_time
print(f"\n🎉 训练完成! 总用时: {total_train_time:.2f} 秒")
print(f"📊 最佳验证F1分数: {best_val_f1:.4f}")

In [ ]:
# ## 9. 可视化训练曲线
print("📈 生成训练曲线...")
curves_save_path = os.path.join(save_dir, f"{cfg['experiment_name']}_training_curves.png")

try:
    visualize_training_curves(training_results, save_path=curves_save_path)
    print(f"✅ 训练曲线已保存: {curves_save_path}")
except Exception as e:
    print(f"❌ 生成训练曲线时出错: {e}")
    
    # 使用matplotlib绘制训练曲线作为备用方案
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 损失曲线
    axes[0].plot(training_results['loss_list'])
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True)
    
    # 准确率曲线
    axes[1].plot(training_results['acc_list'], label='Training')
    axes[1].plot(training_results['val_epoch_list'], training_results['val_acc_list'], label='Validation')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    # F1和Kappa曲线
    axes[2].plot(training_results['val_epoch_list'], training_results['val_f1_macro_list'], 'g-', label='F1 Macro')
    axes[2].plot(training_results['val_epoch_list'], training_results['val_kappa_list'], 'r--', label='Kappa')
    axes[2].set_title('Validation Metrics')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Score')
    axes[2].legend()
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.savefig(curves_save_path)
    plt.show()
    print(f"✅ 使用备用方法生成训练曲线: {curves_save_path}")


In [ ]:
# 单元格10: 在测试集上评估模型（使用项目的评估函数）
print("🧪 开始测试集评估...")

# 加载最佳模型进行测试
if os.path.exists(best_model_path):
    print(f"📂 加载最佳模型: {best_model_path}")
    try:
        # 使用项目的安全加载函数
        from utils.model_io import safe_load_model
        checkpoint = safe_load_model(best_model_path, device)
        
        # 检查检查点格式并加载权重
        if 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
        elif 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            raise KeyError("检查点中未找到模型权重")
            
        print("✅ 成功加载最佳模型权重")
    except Exception as e:
        print(f"⚠️ 加载最佳模型失败: {e}，使用当前模型状态")
else:
    print("⚠️ 未找到最佳模型检查点，使用当前模型状态进行测试")

# 使用项目的完整评估函数 - 与main.py调用方式完全一致
print("\n📊 在测试集上进行完整评估...")
test_results = evaluate_model(
    model=model,
    data_loader=test_loader,
    device=device,
    result_path=save_dir,
    dataset_name="test",
    detailed=True,
    plot=True,
    disable_progress=False,  # 在notebook中显示进度
    show_class_metrics=True
)

print("\n🎯 测试集最终结果:")
print(f"  准确率: {test_results['accuracy']:.4f}")
print(f"  宏平均F1: {test_results['f1_macro']:.4f}")
print(f"  加权F1: {test_results['f1_weighted']:.4f}")
print(f"  平衡准确率: {test_results['balanced_accuracy']:.4f}")
print(f"  Cohen's Kappa: {test_results['kappa']:.4f}")
print(f"  预测类别数: {len(test_results['unique_classes'])}")

# 额外的完整数据集评估（与main.py一致）
print("\n📈 进行完整数据集评估对比...")

# 在训练集上评估
print("\n在训练集上评估...")
train_results = evaluate_model(
    model=model,
    data_loader=train_loader,
    device=device,
    result_path=save_dir,
    dataset_name="train",
    detailed=True,
    plot=True,
    disable_progress=True,
    show_class_metrics=True
)

# 在验证集上评估
print("\n在验证集上评估...")
val_results = evaluate_model(
    model=model,
    data_loader=val_loader,
    device=device,
    result_path=save_dir,
    dataset_name="val",
    detailed=True,
    plot=True,
    disable_progress=True,
    show_class_metrics=True
)

print("\n📊 完整评估结果对比:")
print(f"训练集 - 准确率: {train_results['accuracy']:.4f}, F1: {train_results['f1_macro']:.4f}, Kappa: {train_results['kappa']:.4f}")
print(f"验证集 - 准确率: {val_results['accuracy']:.4f}, F1: {val_results['f1_macro']:.4f}, Kappa: {val_results['kappa']:.4f}")
print(f"测试集 - 准确率: {test_results['accuracy']:.4f}, F1: {test_results['f1_macro']:.4f}, Kappa: {test_results['kappa']:.4f}")

# 使用项目的类别性能比较函数
try:
    from utils.metrics import compare_class_performance
    compare_results = compare_class_performance(
        results_list=[train_results, val_results, test_results],
        dataset_names=["Train", "Validation", "Test"],
        result_path=save_dir
    )
    print("✅ 类别性能比较完成，结果已保存")
except Exception as e:
    print(f"⚠️ 类别性能比较时出错: {e}")

In [ ]:
# 单元格11: 额外的可视化和分析
print("📊 生成额外的可视化结果...")

# 混淆矩阵可视化
try:
    cm_save_path = os.path.join(save_dir, f"{cfg['experiment_name']}_confusion_matrix.png")
    visualize_confusion_matrix(
        test_results['confusion_matrix'], 
        save_path=cm_save_path, 
        log_scale=True
    )
    print(f"✅ 混淆矩阵已保存: {cm_save_path}")
except Exception as e:
    print(f"⚠️ 生成混淆矩阵时出错: {e}")

# 生成类别性能分析
if len(test_results['unique_classes']) > 0:
    print("\n📈 类别性能分析:")
    class_f1_scores = test_results['class_f1']
    unique_classes = test_results['unique_classes']
    
    # 找出表现最好和最差的类别
    best_class_idx = np.argmax(class_f1_scores) if len(class_f1_scores) > 0 else 0
    worst_class_idx = np.argmin(class_f1_scores) if len(class_f1_scores) > 0 else 0
    
    if len(class_f1_scores) > 0:
        print(f"  表现最好的类别: {unique_classes[best_class_idx]} (F1: {class_f1_scores[best_class_idx]:.4f})")
        print(f"  表现最差的类别: {unique_classes[worst_class_idx]} (F1: {class_f1_scores[worst_class_idx]:.4f})")
        print(f"  平均F1分数: {np.mean(class_f1_scores):.4f}")
        print(f"  F1分数标准差: {np.std(class_f1_scores):.4f}")

# 绘制F1分数分布
if len(test_results['class_f1']) > 5:  # 只有足够多的类别时才绘制
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.hist(test_results['class_f1'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    plt.xlabel('F1 Score')
    plt.ylabel('Number of Classes')
    plt.title('Distribution of F1 Scores Across Classes')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    sorted_f1 = np.sort(test_results['class_f1'])
    plt.plot(sorted_f1, 'bo-', markersize=3)
    plt.xlabel('Class Rank (sorted by F1)')
    plt.ylabel('F1 Score')
    plt.title('F1 Scores Sorted by Performance')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    f1_dist_path = os.path.join(save_dir, f"{cfg['experiment_name']}_f1_distribution.png")
    plt.savefig(f1_dist_path)
    plt.show()
    print(f"✅ F1分数分布图已保存: {f1_dist_path}")


In [ ]:
# 单元格12: 保存最终实验结果（与main.py完全一致）
print("💾 保存最终实验结果...")

# 创建与main.py一致的结果结构
final_results = {
    'experiment_info': {
        'experiment_name': cfg['experiment_name'],
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'data_format': 'MAT格式',
        'total_train_time': total_train_time,
        'pytorch_version': torch.__version__,
        'numpy_version': np.__version__,
        'use_bayesian_optimized_params': True
    },
    'hyperparameters_used': {
        'model_type': cfg['model_type'],
        'hidden_units': cfg['hidden_units'],
        'activation': cfg['activation'],
        'dropout_rate': cfg['dropout_rate'],
        'optimizer': cfg['optimizer'],
        'learning_rate': cfg['lr'],
        'weight_decay': cfg['weight_decay'],
        'lr_scheduler_type': cfg['lr_scheduler_type'],
        'lr_step_size': cfg.get('lr_step_size', 4),
        'lr_gamma': cfg.get('lr_gamma', 0.1),
        'batch_size': cfg['batch_size'],
        'epochs': cfg['epochs'],
        'val_epochs': cfg['val_epochs'],
        'random_seed': cfg['random_seed']
    },
    'data_info': {
        'mat_file_path': cfg['mat_file_path'],
        'feature_dim': cfg['feature_dim'],
        'num_classes': cfg['num_class'],
        'train_samples': len(dataset_dict['train_samples']), 
        'val_samples': len(dataset_dict['val_samples']),
        'test_samples': len(dataset_dict['test_samples']),
        'data_processing': '背景像素已在加载阶段过滤',
        'normalization': 'StandardScaler应用于特征',
        'patient_split': cfg['dataset_split']
    },
    'training_results': training_results,
    'final_performance': {
        'best_validation_f1': best_val_f1,
        # 训练集性能
        'train_accuracy': train_results['accuracy'],
        'train_f1_macro': train_results['f1_macro'],
        'train_f1_weighted': train_results['f1_weighted'],
        'train_balanced_accuracy': train_results['balanced_accuracy'],
        'train_kappa': train_results['kappa'],
        # 验证集性能
        'val_accuracy': val_results['accuracy'],
        'val_f1_macro': val_results['f1_macro'],
        'val_f1_weighted': val_results['f1_weighted'],
        'val_balanced_accuracy': val_results['balanced_accuracy'],
        'val_kappa': val_results['kappa'],
        # 测试集性能
        'test_accuracy': test_results['accuracy'],
        'test_f1_macro': test_results['f1_macro'],
        'test_f1_weighted': test_results['f1_weighted'],
        'test_balanced_accuracy': test_results['balanced_accuracy'],
        'test_kappa': test_results['kappa'],
        'predicted_classes_count': len(test_results['unique_classes'])
    },
    'bayesian_optimization_info': {
        'original_hyperparams': best_hyperparams_from_bayesian,
        'source': '贝叶斯优化得到的最佳超参数',
        'optimization_metric': 'F1_macro'
    }
}

# 保存为JSON - 与main.py一致
results_json_path = os.path.join(save_dir, f"{cfg['experiment_name']}_complete_results.json")
with open(results_json_path, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=4, ensure_ascii=False)
print(f"✅ 完整结果已保存: {results_json_path}")

# 保存实验配置 - 与main.py一致
config_save_path = os.path.join(save_dir, f"{cfg['experiment_name']}_config.json")
config_to_save = cfg.copy()
# 移除不可序列化的对象
if 'scaler' in config_to_save:
    del config_to_save['scaler']
if 'pca_model' in config_to_save:
    del config_to_save['pca_model']

save_config(config_to_save, config_save_path)
print(f"✅ 实验配置已保存: {config_save_path}")

# 创建最终报告文本 - 与main.py格式完全一致
final_report_path = os.path.join(save_dir, f"{cfg['experiment_name']}_final_report.txt")
with open(final_report_path, 'w', encoding='utf-8') as f:
    f.write(f"脑体素分类实验最终报告\n")
    f.write(f"{'='*50}\n\n")
    f.write(f"实验名称: {cfg['experiment_name']}\n")
    f.write(f"完成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"数据格式: MAT格式\n")
    f.write(f"训练时长: {total_train_time:.2f} 秒\n")
    f.write(f"使用贝叶斯优化最佳超参数: 是\n\n")
    
    f.write(f"模型架构: {cfg['model_type']}\n")
    f.write(f"隐藏层配置: {cfg['hidden_units']}\n")
    f.write(f"激活函数: {cfg['activation']}\n")
    f.write(f"Dropout率: {cfg['dropout_rate']}\n")
    f.write(f"优化器: {cfg['optimizer']}\n")
    f.write(f"学习率: {cfg['lr']}\n")
    f.write(f"权重衰减: {cfg['weight_decay']}\n\n")
    
    f.write(f"数据信息:\n")
    f.write(f"特征维度: {cfg['feature_dim']}\n")
    f.write(f"类别数量: {cfg['num_class']}\n")
    f.write(f"训练样本: {len(dataset_dict['train_samples'])}\n")
    f.write(f"验证样本: {len(dataset_dict['val_samples'])}\n")
    f.write(f"测试样本: {len(dataset_dict['test_samples'])}\n")
    f.write(f"数据处理: 背景像素已在加载阶段过滤\n\n")
    
    f.write(f"训练集性能:\n")
    f.write(f"  准确率: {train_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {train_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {train_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {train_results['kappa']:.4f}\n\n")
    
    f.write(f"验证集性能:\n")
    f.write(f"  准确率: {val_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {val_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {val_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {val_results['kappa']:.4f}\n\n")
    
    f.write(f"测试集性能:\n")
    f.write(f"  准确率: {test_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {test_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {test_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {test_results['kappa']:.4f}\n\n")
    
    f.write(f"最佳模型保存路径: {best_model_path}\n")
    f.write(f"所有结果保存目录: {save_dir}\n")

print(f"✅ 最终报告已保存: {final_report_path}")

# 保存评估结果摘要 - 与main.py一致的格式
summary_path = os.path.join(save_dir, "evaluation_summary.txt")
with open(summary_path, 'w') as f:
    f.write("评估结果摘要\n")
    f.write("="*50 + "\n\n")
    
    f.write(f"数据格式: MAT格式\n")
    f.write(f"数据处理: 背景像素已在加载阶段过滤\n")
    f.write(f"使用贝叶斯优化超参数: 是\n\n")
    
    f.write("训练集结果:\n")
    f.write(f"  准确率: {train_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {train_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {train_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {train_results['kappa']:.4f}\n\n")
    
    f.write("验证集结果:\n")
    f.write(f"  准确率: {val_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {val_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {val_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {val_results['kappa']:.4f}\n\n")
    
    f.write("测试集结果:\n")
    f.write(f"  准确率: {test_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {test_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {test_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {test_results['kappa']:.4f}\n\n")
    
    f.write(f"模型: {cfg['model_type']}\n")
    f.write(f"隐藏层: {cfg['hidden_units']}\n")
    f.write(f"激活函数: {cfg['activation']}\n")
    f.write(f"Dropout率: {cfg['dropout_rate']}\n")
    f.write(f"优化器: {cfg['optimizer']}\n")
    f.write(f"学习率: {cfg['lr']}\n")
    f.write(f"权重衰减: {cfg['weight_decay']}\n")

print(f"✅ 评估摘要已保存: {summary_path}")

# 打印实验总结 - 与main.py格式一致
print(f"\n🎉 实验完成总结:")
print(f"  📁 结果目录: {save_dir}")
print(f"  🏆 最佳验证F1: {best_val_f1:.4f}")
print(f"  🎯 测试集F1: {test_results['f1_macro']:.4f}")
print(f"  🎯 测试集准确率: {test_results['accuracy']:.4f}")
print(f"  🎯 测试集Kappa: {test_results['kappa']:.4f}")
print(f"  ⏱️ 总训练时间: {total_train_time:.2f} 秒")
print(f"  📊 使用了贝叶斯优化的最佳超参数")
print(f"  📈 所有可视化和详细结果已保存")
print(f"  🏆 最佳模型: {os.path.basename(best_model_path)}")

print("\n✨ Notebook执行完毕!")